In [23]:
import json
import glob
import os
from eval_utils import extract_decision, get_eval_accuracy
from datasets import load_dataset

def parse_checkpoint(f):
    # Assuming format like "prefix_ckpt{idx}_suffix"
    base = f.split('ckpt')[0].strip('_')  # Get the prefix before 'ckpt'
    remaining = f.split('ckpt')[1]
    
    # Split the remaining part into idx and suffix
    idx = int(remaining.split('_')[0])  # Extract the number after 'ckpt'
    suffix = remaining.split('_')[1] if '_' in remaining else ''  # Get suffix if exists
    suffix = suffix.strip('.json')
    return base, idx, suffix

# glob all files in pub-med-eval
subset = "labeled"
files = glob.glob(f"pub-med-eval-v2/cpt/*_subset={subset}.json")
true_data = load_dataset("qiaojin/PubMedQA", f"pqa_{subset}", split="train")


In [39]:
def evaluate_pubmed_predictions(file_path, true_data):
    """
    Evaluate model predictions against ground truth for PubMedQA dataset.
    
    Args:
        file_path: Path to the JSON file containing model generations
        true_data: Dataset containing ground truth decisions
        
    Returns:
        float: Accuracy of the predictions
    """
    # Load predictions
    pred_data = json.load(open(file_path, 'r'))
    generations = pred_data["generations_str"]
    
    # Extract decisions from generations
    def extract_decision_text(text):
        if not text:
            return None

        text = text.rstrip('#').lower()
        if text.endswith('yes'):
            return 'yes'
        elif text.endswith('no'):
            return 'no'
        elif text.endswith('maybe'):
            return 'maybe'

        try:
            if (s := 'final decision') in text:
                decision = text.split(s)[1]
            elif (s := 'final decision is') in text:
                decision = text.split(s)[1]
            else:
                return None
                
            return decision
        except Exception as e:
            print(f"Error: {e}\n{text}\n")
            return None
    
    # Process all generations
    pred_decisions = []
    for generation in generations:
        decision = extract_decision_text(generation)
        if decision is None:
            print(f"Error extracting decision from: \n{generation}\n\n")
        pred_decisions.append(decision)
    
    # Get ground truth decisions
    true_decisions = true_data["final_decision"]
    
    # Calculate and return accuracy
    accuracy = get_eval_accuracy(pred_decisions, true_decisions)
    return accuracy

# Evaluate the first file
for f in files:
    accuracy = evaluate_pubmed_predictions(f, true_data)
    print(f)
    print(f"Accuracy: {accuracy:.4f}")


pub-med-eval-v2/cpt/mora_cpt_witheval_ckpt3200_subset=labeled.json
Accuracy: 0.3820
pub-med-eval-v2/cpt/mora_cpt_witheval_ckpt6400_subset=labeled.json
Accuracy: 0.4130
pub-med-eval-v2/cpt/mora_cpt_witheval_ckpt9600_subset=labeled.json
Accuracy: 0.4070
